# Bank Association Rule Mining — Product Affinity

## Business Case

A bank wants to understand:

> **Which banking products tend to be owned or used together by the same customers?**

Examples:

```text
Savings Account + Mobile Banking
Credit Card + Personal Loan
Investment + Time Deposit
Debit Card + E-Wallet
```

Association Rule Mining can discover relationships such as:

```text
Customers with A
        ↓
are also likely to have B
```

This can support:

- Cross-sell analysis
- Product affinity
- Campaign design
- Next Best Product candidates
- Bundle discovery
- Customer journey analysis

> This notebook uses synthetic customer-product data for educational purposes.

## 1. Association Rule Mining

Association Rule Mining finds relationships among items.

A rule looks like:

```text
A → B
```

Example:

```text
Mobile Banking → Debit Card
```

This does **not automatically mean A causes B**.

It means customers who have A may have B more frequently than expected.

## 2. Key Metrics

### Support

How frequently the combination appears in the dataset.

```text
Support(A → B)
= P(A and B)
```

### Confidence

Among customers who have A, how many also have B?

```text
Confidence(A → B)
= P(B | A)
```

### Lift

How much stronger the relationship is compared with B occurring independently.

```text
Lift(A → B)
= P(B | A) / P(B)
```

Interpretation:

```text
Lift > 1 → positive association
Lift = 1 → approximately independent
Lift < 1 → negative association
```

Lift is often more informative than confidence when B is already very common.

## 3. Banking Interpretation

Suppose:

```text
Rule:
Credit Card → Personal Loan

Support     = 12%
Confidence  = 40%
Lift        = 1.8
```

Interpretation:

- 12% of customers have both products.
- 40% of customers with Credit Card also have Personal Loan.
- The presence of Credit Card is associated with Personal Loan at a rate 1.8× the baseline occurrence of Personal Loan.

This is an **association**, not proof that Credit Card ownership causes Personal Loan ownership.

## 4. Import Libraries

This step imports the libraries used throughout the notebook — pandas and NumPy for data handling, mlxtend for Apriori and association rules, and Matplotlib/Seaborn for visualisation.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns",100)

df=pd.read_csv("bank_product_baskets_sample.csv")

print("Rows:",len(df))
print("Customers:",df["Customer_ID"].nunique())
print("Products:",df["Product"].nunique())

display(df.head())

## 5. Dataset Dictionary

The basket data is documented: customer, product and latent group. Association mining needs transactions grouped per customer, so the dictionary fixes the grain the baskets will be built on.


In [ ]:
dictionary=pd.DataFrame({
    "Column":["Customer_ID","Product","Latent_Group"],
    "Meaning":[
        "Customer identifier",
        "Banking product owned by customer",
        "Synthetic latent segment used only to generate sample data"
    ]
})
display(dictionary)

## 6. Product Frequency

Product penetration — the share of customers holding each product — is calculated. Very rare or near-universal products distort support-based mining, so this view guides filtering before rule generation.


In [ ]:
product_frequency=(
    df.groupby("Product")["Customer_ID"]
      .nunique()
      .sort_values(ascending=False)
      .reset_index(name="Customers")
)

product_frequency["Share"]=(
    product_frequency["Customers"]/df["Customer_ID"].nunique()
)

display(product_frequency.round(3))

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(
    data=product_frequency,
    y="Product",
    x="Share"
)
plt.title("Banking Product Penetration")
plt.xlabel("Customer Share")
plt.ylabel("")
plt.show()

## 7. Create Customer Baskets

Association Rule Mining requires transactions/baskets.

Each customer becomes one basket:

```text
Customer A:
[
 Savings Account,
 Debit Card,
 Mobile Banking
]
```

Customer B:

```text
[
 Savings Account,
 Credit Card,
 Personal Loan
]
```

In [ ]:
baskets=(
    df.groupby("Customer_ID")["Product"]
      .apply(list)
      .reset_index(name="Basket")
)

display(baskets.head())

## 8. Convert Baskets to One-Hot Matrix

Association algorithms commonly expect:

```text
Customer | Savings | Debit | Credit Card | Loan | ...
-------------------------------------------------------
C00001   |    1    |   1   |      0      |  0   |
C00002   |    1    |   1   |      1      |  1   |
```

1 means the product exists in the customer's basket.

In [ ]:
te=TransactionEncoder()

basket_matrix=te.fit(baskets["Basket"]).transform(
    baskets["Basket"]
)

basket_df=pd.DataFrame(
    basket_matrix,
    columns=te.columns_,
    index=baskets["Customer_ID"]
)

display(basket_df.head())
print("Matrix shape:",basket_df.shape)

## 9. Apriori Algorithm

The Apriori algorithm finds itemsets of products that occur together above a minimum support threshold. It is the engine that produces the candidate patterns from which the association rules are derived.


Apriori searches for **frequent itemsets**.

Example:

```text
{Savings Account}
{Mobile Banking}
{Savings Account, Mobile Banking}
{Credit Card, Personal Loan}
...
```

We specify a minimum support threshold to avoid generating an enormous number of rare combinations.

In [ ]:
frequent_itemsets=apriori(
    basket_df,
    min_support=0.05,
    use_colnames=True
)

frequent_itemsets["itemset_size"] = (
    frequent_itemsets["itemsets"].apply(len)
)

frequent_itemsets=frequent_itemsets.sort_values(
    "support",
    ascending=False
)

display(frequent_itemsets.head(20))

## 10. Frequent Itemsets

An itemset:

```text
{Savings Account, Debit Card}
```

with support:

```text
0.65
```

means 65% of customer baskets contain both products.

Frequent itemsets are the building blocks for association rules.

In [ ]:
plt.figure(figsize=(9,5))
sns.histplot(
    frequent_itemsets["support"],
    bins=20
)
plt.title("Distribution of Frequent Itemset Support")
plt.xlabel("Support")
plt.show()

## 11. Generate Association Rules

Now transform frequent itemsets into directional rules:

```text
A → B
```

A rule has:

- antecedent = A
- consequent = B

In [ ]:
rules=association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=1.10
)

rules=rules.sort_values(
    ["lift","confidence","support"],
    ascending=False
)

display(rules[[
    "antecedents","consequents",
    "support","confidence","lift"
]].head(20))

## 12. Clean Rule Output

For business users, convert Python sets into readable text.

In [ ]:
def itemset_to_text(x):
    return " + ".join(sorted(x))

rules_clean=rules.copy()

rules_clean["Antecedent"]=rules_clean["antecedents"].apply(itemset_to_text)
rules_clean["Consequent"]=rules_clean["consequents"].apply(itemset_to_text)

rules_clean=rules_clean[[
    "Antecedent","Consequent",
    "support","confidence","lift"
]].rename(columns={
    "support":"Support",
    "confidence":"Confidence",
    "lift":"Lift"
})

display(rules_clean.head(20).round(3))

## 13. How to Read a Rule

Consider:

```text
Savings Account → Mobile Banking
```

### Support

How common is the combination?

### Confidence

If a customer has Savings Account, how often do they also have Mobile Banking?

### Lift

How much stronger is this relationship compared with the baseline probability of Mobile Banking?

A high lift indicates a relatively strong association, but a rule with tiny support may not be operationally useful.

## 14. Filter for Practical Rules

A bank usually does not want every mathematical rule.

We can require:

```text
Support ≥ 5%
Confidence ≥ 20%
Lift ≥ 1.20
```

The thresholds are examples and should be determined by business scale, campaign capacity, validation results, and risk considerations.

In [ ]:
practical_rules=rules_clean[
    (rules_clean["Support"]>=0.05) &
    (rules_clean["Confidence"]>=0.20) &
    (rules_clean["Lift"]>=1.20)
].sort_values(
    ["Lift","Confidence","Support"],
    ascending=False
)

display(practical_rules.head(20).round(3))

## 15. Product Affinity Heatmap

Another useful view is a pairwise affinity matrix.

For each product pair, calculate:

```text
P(B | A)
```

This answers:

> Among customers who have product A, how frequently do they also have product B?

In [ ]:
products_sorted=product_frequency["Product"].tolist()

affinity=pd.DataFrame(
    index=products_sorted,
    columns=products_sorted,
    dtype=float
)

for a in products_sorted:
    for b in products_sorted:
        affinity.loc[a,b]=basket_df[
            basket_df[a]
        ][b].mean()

plt.figure(figsize=(12,9))
sns.heatmap(
    affinity,
    annot=True,
    fmt=".2f",
    cmap="Blues"
)
plt.title("Product Affinity — P(B | A)")
plt.xlabel("Product B")
plt.ylabel("Product A")
plt.show()

## 16. Cross-Sell Opportunities

A useful business interpretation is:

```text
Customer has A
      ↓
Customer does not have B
      ↓
A → B is a strong association
      ↓
Candidate for cross-sell
```

However, association alone should not automatically trigger a campaign.

Additional checks should include:

- eligibility,
- customer consent,
- affordability,
- credit policy,
- campaign fatigue,
- product suitability,
- profitability,
- compliance.

## 17. Build a Cross-Sell Candidate Table

Simple one-product → one-product rules are turned into a cross-sell table: customers who own the antecedent product but not the consequent become concrete, nameable campaign targets.


In [ ]:
# Select one-item antecedent and one-item consequent rules
cross_sell=rules_clean[
    rules_clean["Antecedent"].str.count(r"\+")==0
].copy()

cross_sell=cross_sell[
    cross_sell["Consequent"].str.count(r"\+")==0
].copy()

cross_sell=cross_sell.sort_values(
    ["Lift","Confidence","Support"],
    ascending=False
)

display(cross_sell.head(20).round(3))

## 18. Example Customer Recommendation Logic

Association Rule Mining can be used as one component of a recommendation pipeline.

Example:

```text
Customer owns:
Savings Account
Debit Card
Mobile Banking

Rules:
Savings Account → Investment
Debit Card → Credit Card

Candidate products:
Investment
Credit Card
```

The recommendation layer can then combine:

```text
Affinity
+
Eligibility
+
Customer Need
+
Risk
+
Business Rules
```

This is different from simply recommending the product with the highest lift.

In [ ]:
customer_products=set([
    "Savings Account",
    "Debit Card",
    "Mobile Banking"
])

candidate_rules=rules_clean[
    rules_clean["Antecedent"].isin(customer_products)
].copy()

candidate_rules=candidate_rules[
    ~candidate_rules["Consequent"].isin(customer_products)
]

display(
    candidate_rules.sort_values(
        ["Lift","Confidence"],
        ascending=False
    ).head(10).round(3)
)

## 19. Association Rules vs Recommendation

Association Rule Mining:

> "What products frequently occur together?"

Recommendation:

> "What product should I recommend to this particular customer?"

Association rules can therefore be used as an **input** to a recommendation engine, but they are not the complete recommendation system.

## 20. Association Rules vs Classification

Classification:

```text
Will customer buy product B?
```

Association:

```text
What products are commonly associated?
```

A classification model uses a target variable.

Association Rule Mining does not require a traditional target variable.

## 21. Association Rules vs Clustering

Clustering:

```text
Which customers are similar?
```

Association:

```text
Which products occur together?
```

They can also be combined:

```text
Customer Segmentation
        ↓
Run Association Rules per Segment
        ↓
Segment-specific Product Affinity
```

This often produces more actionable insights than a single rule model across all customers.

## 22. Segment-Level Association Mining

For a real bank, you may calculate rules separately for:

- Mass customers
- Affluent customers
- Priority customers
- Digital customers
- Payroll customers
- SME customers

This can reveal that a product relationship is strong in one segment but weak in another.

In [ ]:
segment_baskets=(
    df.groupby(["Latent_Group","Customer_ID"])["Product"]
      .apply(list)
      .reset_index(name="Basket")
)

segment_rule_summary=[]

for segment in segment_baskets["Latent_Group"].unique():
    subset=segment_baskets[
        segment_baskets["Latent_Group"]==segment
    ]

    te_s=TransactionEncoder()
    mat_s=te_s.fit(subset["Basket"]).transform(subset["Basket"])
    basket_s=pd.DataFrame(mat_s,columns=te_s.columns_)

    fi_s=apriori(
        basket_s,
        min_support=0.10,
        use_colnames=True
    )

    if len(fi_s)>0:
        r_s=association_rules(
            fi_s,
            metric="lift",
            min_threshold=1.10
        )

        if len(r_s)>0:
            r_s["Segment"]=segment
            segment_rule_summary.append(r_s)

segment_rules=pd.concat(segment_rule_summary,ignore_index=True)

display(
    segment_rules[[
        "Segment","antecedents",
        "consequents","support",
        "confidence","lift"
    ]].sort_values(
        "lift",ascending=False
    ).head(20)
)

## 23. Business Dashboard Ideas

A Product Affinity dashboard can contain:

### KPI

- Customers
- Products
- Frequent Itemsets
- Association Rules
- High-Lift Rules

### Visuals

1. Product penetration
2. Product affinity heatmap
3. Support vs confidence scatter
4. Top association rules
5. Segment-level affinity
6. Product network

A business user should be able to filter by customer segment and product.

In [ ]:
plt.figure(figsize=(10,7))
plot_rules=rules_clean[
    (rules_clean["Support"]>=0.03)
].copy()

sns.scatterplot(
    data=plot_rules,
    x="Support",
    y="Confidence",
    size="Lift",
    hue="Lift",
    sizes=(30,300)
)

plt.title("Association Rules: Support vs Confidence")
plt.show()

## 24. Network View Concept

Association rules can also be represented as a graph:

```text
Savings ───────→ Investment
   │
   └───────────→ Mobile Banking

Credit Card ───→ Personal Loan
```

- Nodes = products
- Edges = associations
- Edge weight = lift / confidence

Graph visualization becomes especially useful when the number of products grows.

## 25. Production Architecture

```text
Customer Product Data
        ↓
Data Warehouse
        ↓
Customer Basket
        ↓
Frequent Itemset Mining
        ↓
Association Rules
        ↓
Business Filtering
        ↓
Eligibility / Compliance
        ↓
Recommendation Engine
        ↓
CRM / Mobile Banking
        ↓
Customer Response
        ↓
Campaign Measurement
```

For production, rule generation may run periodically while recommendation serving can happen in batch or near real-time.

## 26. Important Banking Controls

Association does not equal causation.

A strong rule should not automatically mean:

> "Give this product to every customer."

Before deployment, validate:

- customer eligibility,
- suitability,
- affordability,
- credit risk,
- consent,
- privacy,
- marketing frequency,
- fairness,
- product profitability,
- campaign performance.

Association rules should support decision-making, not replace banking controls.

## 27. Common Mistakes

1. Using confidence alone.
2. Ignoring lift.
3. Selecting rules with extremely low support.
4. Treating association as causation.
5. Generating too many rules without business filtering.
6. Ignoring customer segments.
7. Recommending products without eligibility checks.
8. Ignoring temporal order.
9. Using future product ownership to recommend past actions.
10. Not validating rules on a holdout period.

## 28. Temporal Validation

A stronger production approach is:

```text
Historical Period
       ↓
Generate Rules
       ↓
Future Holdout Period
       ↓
Test Whether Associations Persist
```

For example:

```text
Train:
January–September

Validation:
October–November

Test:
December
```

This helps detect rules that are only artifacts of a particular period.

## 29. Key Evaluation Metrics

### Statistical

- Support
- Confidence
- Lift

### Business

- Cross-sell conversion
- Incremental revenue
- Campaign acceptance
- Product activation
- Customer engagement

### Risk / Governance

- Complaint rate
- Opt-out rate
- Eligibility violations
- Unwanted targeting
- Segment disparities

## 30. Final Executive Summary

### Business Question

> **Which banking products are commonly associated with each other?**

### Method

```text
Customer Product Data
        ↓
Basket Construction
        ↓
One-Hot Encoding
        ↓
Apriori
        ↓
Frequent Itemsets
        ↓
Association Rules
        ↓
Support / Confidence / Lift
        ↓
Business Filtering
        ↓
Cross-Sell / Recommendation Candidate
```

### Main Insight

Association Rule Mining discovers **product affinity**.

It answers:

> **"Customers who have A also tend to have B."**

It does not by itself answer:

> **"Should we offer B to this customer?"**

For that, combine affinity with customer-level prediction, eligibility, risk, business rules, and experimentation.